# 02 - Data Preprocessing

This notebook cleans the DDXPlus raw split from `app/data/raw/ddxplus`, parses the evidence-token grammar, and prepares a machine-learning-ready table for the next notebooks.

In [1]:
import sys
from pathlib import Path

_nb = Path.cwd().resolve()
for _c in [_nb, _nb.parent, _nb.parent.parent]:
    if (_c / "app" / "main.py").exists():
        sys.path.insert(0, str(_c))
        break

from training.notebooks.bootstrap import require_ddxplus

ctx = require_ddxplus()
repo_root = ctx.repo_root
raw_path = ctx.raw_train_csv
processed_dir = ctx.processed_dir
training_models_dir = ctx.training_models_dir
runtime_models_dir = ctx.runtime_models_dir

import ast
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer

usecols = ["AGE", "DIFFERENTIAL_DIAGNOSIS", "SEX", "PATHOLOGY", "EVIDENCES", "INITIAL_EVIDENCE"]
df = pd.read_csv(raw_path, usecols=usecols)
print("Loaded:", raw_path)
print("Before shape:", df.shape)
print("Columns:", list(df.columns))


Loaded: C:\healix-ai-medical-assistant\app\data\raw\ddxplus\train.csv
Before shape: (20000, 6)
Columns: ['AGE', 'DIFFERENTIAL_DIAGNOSIS', 'SEX', 'PATHOLOGY', 'EVIDENCES', 'INITIAL_EVIDENCE']


In [2]:
# Parse and clean the evidence-token grammar

def split_token(token: str) -> str:
    return str(token).split("_@_", 1)[0]


def parse_evidences(value):
    text = "[]" if pd.isna(value) else str(value)
    try:
        tokens = ast.literal_eval(text)
    except (ValueError, SyntaxError):
        tokens = []
    return [split_token(token) for token in tokens]


df["evidence_tokens"] = df["EVIDENCES"].apply(parse_evidences)

def encode_sex(value):
    text = str(value).strip().upper()
    return 1 if text == "M" else 0


df["SEX_M"] = df["SEX"].apply(encode_sex)
df["AGE"] = pd.to_numeric(df["AGE"], errors="coerce")

print(df[["AGE", "SEX", "PATHOLOGY", "INITIAL_EVIDENCE", "evidence_tokens"]].head(5).to_string())

   AGE SEX                PATHOLOGY INITIAL_EVIDENCE                                                                                                                                                                                                                 evidence_tokens
0   18   M                     URTI             E_91                                                                                                           [E_48, E_50, E_53, E_54, E_54, E_55, E_55, E_55, E_56, E_57, E_58, E_59, E_77, E_79, E_91, E_97, E_201, E_204, E_222]
1   21   M  HIV (initial infection)             E_50                      [E_9, E_27, E_50, E_51, E_53, E_54, E_55, E_55, E_55, E_56, E_57, E_58, E_59, E_91, E_115, E_129, E_130, E_131, E_132, E_133, E_133, E_133, E_133, E_133, E_134, E_135, E_136, E_148, E_162, E_189, E_204]
2   19   F                Pneumonia             E_77  [E_53, E_54, E_54, E_55, E_55, E_55, E_55, E_55, E_56, E_57, E_58, E_59, E_77, E_78, E_79, E_94, E_106, E_118, E_12

In [3]:
# Before/after example for a sample row
sample = df.iloc[0]
print("Before:")
print("PATHOLOGY:", sample["PATHOLOGY"])
print("SEX:", sample["SEX"])
print("EVIDENCES:", sample["EVIDENCES"])

print("\nAfter:")
print("AGE:", sample["AGE"])
print("SEX_M:", sample["SEX_M"])
print("Evidence tokens:", sample["evidence_tokens"][:10])

Before:
PATHOLOGY: URTI
SEX: M
EVIDENCES: ['E_48', 'E_50', 'E_53', 'E_54_@_V_161', 'E_54_@_V_183', 'E_55_@_V_89', 'E_55_@_V_108', 'E_55_@_V_167', 'E_56_@_4', 'E_57_@_V_123', 'E_58_@_3', 'E_59_@_3', 'E_77', 'E_79', 'E_91', 'E_97', 'E_201', 'E_204_@_V_10', 'E_222']

After:
AGE: 18
SEX_M: 1
Evidence tokens: ['E_48', 'E_50', 'E_53', 'E_54', 'E_54', 'E_55', 'E_55', 'E_55', 'E_56', 'E_57']


In [4]:
# Convert evidence tokens to binary evidence columns
all_evidence_codes = sorted({code for tokens in df["evidence_tokens"] for code in tokens})
mlb = MultiLabelBinarizer(classes=all_evidence_codes)
evidence_matrix = mlb.fit_transform(df["evidence_tokens"])
evidence_columns = list(mlb.classes_)
evidence_df = pd.DataFrame(evidence_matrix, columns=evidence_columns, index=df.index).astype(int)

print("Evidence features created:", len(evidence_columns))
print(evidence_df.head().iloc[:, :10])

Evidence features created: 223
   E_0  E_1  E_10  E_100  E_101  E_102  E_103  E_104  E_105  E_106
0    0    0     0      0      0      0      0      0      0      0
1    0    0     0      0      0      0      0      0      0      0
2    0    0     0      0      0      0      0      0      0      1
3    0    0     0      0      0      0      0      0      0      0
4    0    0     0      0      0      0      0      0      0      0


In [5]:
# Encode target pathology labels
label_encoder = LabelEncoder()
df["target"] = label_encoder.fit_transform(df["PATHOLOGY"])
print("Classes:", list(label_encoder.classes_[:10]), "...")
print("Label count:", len(label_encoder.classes_))

Classes: ['Acute COPD exacerbation / infection', 'Acute dystonic reactions', 'Acute laryngitis', 'Acute otitis media', 'Acute pulmonary edema', 'Acute rhinosinusitis', 'Allergic sinusitis', 'Anaphylaxis', 'Anemia', 'Atrial fibrillation'] ...
Label count: 49


In [6]:
# Build the feature matrix
feature_columns = ["AGE", "SEX_M"] + evidence_columns
X = pd.concat([df[["AGE", "SEX_M"]].reset_index(drop=True), evidence_df.reset_index(drop=True)], axis=1)
y = df["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("First feature row:")
print(X.iloc[0].head(10).to_dict())

X shape: (20000, 225)
y shape: (20000,)
First feature row:
{'AGE': 18, 'SEX_M': 1, 'E_0': 0, 'E_1': 0, 'E_10': 0, 'E_100': 0, 'E_101': 0, 'E_102': 0, 'E_103': 0, 'E_104': 0}


In [ ]:
# Save processed data for later notebooks
out_dir = repo_root / "training" / "data" / "processed"
out_dir.mkdir(parents=True, exist_ok=True)
processed_path = out_dir / "processed_dataset.csv"
processed_df = pd.concat(
    [df[["AGE", "SEX", "SEX_M", "PATHOLOGY", "target", "INITIAL_EVIDENCE"]].reset_index(drop=True), evidence_df.reset_index(drop=True)],
    axis=1,
)
processed_df.to_csv(processed_path, index=False)
print("Processed dataset saved to:", processed_path)

# Save label encoder and feature names to the runtime location expected by
# app/infrastructure/model_loader.py
runtime_models_dir = repo_root / "models"
runtime_models_dir.mkdir(parents=True, exist_ok=True)

feature_names = feature_columns
runtime_feature_names_path = runtime_models_dir / "feature_names.json"
runtime_label_encoder_path = runtime_models_dir / "label_encoder.joblib"
runtime_feature_names_path.write_text(json.dumps(feature_names, indent=2), encoding="utf-8")
joblib.dump(label_encoder, runtime_label_encoder_path)

print("Feature names saved to:", runtime_feature_names_path)
print("Label encoder saved to:", runtime_label_encoder_path)

In [8]:
# Verify that the exported artifacts can be loaded back successfully
reloaded_feature_names = json.loads(runtime_feature_names_path.read_text(encoding="utf-8"))
reloaded_label_encoder = joblib.load(runtime_label_encoder_path)

print("Saved feature count:", len(reloaded_feature_names))
print("Saved class count:", len(reloaded_label_encoder.classes_))
print("Feature names match:", reloaded_feature_names == feature_columns)
print("First feature names:", reloaded_feature_names[:10])


Saved feature count: 225
Saved class count: 49
Feature names match: True
First feature names: ['AGE', 'SEX_M', 'E_0', 'E_1', 'E_10', 'E_100', 'E_101', 'E_102', 'E_103', 'E_104']


## Preprocessing summary

- The raw DDXPlus split was loaded from `app/data/raw/ddxplus/train.csv`.
- Evidence strings were parsed into token lists and reduced to base evidence codes.
- Each evidence code became a binary feature column.
- `PATHOLOGY` was encoded into numeric class labels for classification.
- The processed table and encoders were saved for the next notebooks.